# v0.23.0 — Rich field types

## 1. Intro

SurrealDB has value types that JSON does not: datetimes, durations, decimals, UUIDs, bytes,
ranges. Before v0.23.0 a `timedelta`, a `date` or a `set` field could not be written, and a stored
duration could not be read into a `timedelta` field. Since v0.23.0:

* `datetime`, `date`, `timedelta`, `Decimal`, `UUID`, `bytes` and `set[X]`/`frozenset[X]` fields are
  stored as SurrealDB's own values by every write, and read back as the declared type.
* Two ORM types cover what Python has no type for: `Duration`, a duration with SurrealDB's
  nanosecond precision, and `Range[T]`, a range whose bounds are validated as `T`.
* Filter values are sent like field values: `filter(trial__gte=timedelta(days=7))` compares
  durations, and `filter(hour__in=Range(...))` tests membership in a range. Live queries write the
  same values into their statement as SurrealQL literals.
* A value SurrealDB cannot hold (a negative duration, an integer beyond 64 bits, a decimal beyond
  `2**96 - 1`) is refused with a `ValueError` before anything is sent.

The ORM behaves the same on SurrealDB 2.x and 3.x: this notebook runs top to bottom on either. The
one difference is a column declared `TYPE set<…>` in a schema (section 9): declare it `array<…>`.

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")

# WebSocket: the live query of section 6 needs it.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. Models

`Subscription` uses every rich annotation. No import is needed for the Python types; `Duration`,
`Range` and the bound classes `BoundIncluded`/`BoundExcluded` come from `surreal_orm_lite` (the
bound classes are the SDK's). `Slot` has a plain `hour: int`, for range membership in section 5.

The tables are schemaless, so `tags` is stored as an array. The notebook starts by dropping its
tables, so it can be re-executed in place; on SurrealDB 3.x removing a missing table raises, hence
the suppressed errors.

In [2]:
from datetime import UTC, date, datetime, timedelta
from decimal import Decimal
from uuid import UUID

from surreal_orm_lite import (
    BaseSurrealModel,
    BoundExcluded,
    BoundIncluded,
    Duration,
    Range,
    SurrealConfigDict,
)


class Subscription(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_subscription")

    id: str | None = None
    started_at: datetime
    renews_on: date
    trial: timedelta
    grace: Duration | None = None
    price: Decimal
    token: UUID
    active_hours: Range[int]
    period: Range[datetime] | None = None
    logo: bytes | None = None
    tags: set[str] = set()


class Slot(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_slot")

    id: str | None = None
    hour: int


client = await SurrealDBConnectionManager.get_client()
DEMO_TABLES = ["nb_subscription", "nb_slot"]


async def reset() -> None:
    """Drop every table of this notebook, whether or not it exists."""
    for table in DEMO_TABLES:
        with contextlib.suppress(Exception):
            await client.query(f"REMOVE TABLE {table};", {})


await reset()
print("Demo tables reset")

Demo tables reset


## 3. Save and read back

Each value is stored as the server's own type, and `get()` returns the declared Python type: a
`date` is a `datetime` at midnight UTC on the server and a `date` again here, a `timedelta` is a
SurrealDB `duration`, a `set` an array (sorted when its items can be ordered). A `Range` is a
SurrealDB range; an open side, `None`, is sent as SurrealDB's null bound (section 5 saves one).

In [3]:
BASIC_TOKEN = UUID("0190c3d1-5c3b-7cc6-8f8a-2d2b1f0e9a11")


def type_name(value: object) -> str:
    """`package.Class` of a value: `surreal_orm_lite.Duration` is the ORM's, `surrealdb.Duration` the SDK's."""
    return f"{type(value).__module__.split('.')[0]}.{type(value).__name__}"


await Subscription(
    id="basic",
    started_at=datetime(2026, 1, 2, 9, 30, tzinfo=UTC),
    renews_on=date(2026, 2, 2),
    trial=timedelta(days=7),
    grace=Duration.parse("1h30m1ns"),  # nanoseconds kept
    price=Decimal("9.99"),
    token=BASIC_TOKEN,
    active_hours=Range(BoundIncluded(9), BoundExcluded(17)),  # 9..17: 9 included, 17 excluded
    period=Range(BoundIncluded(datetime(2026, 1, 1, tzinfo=UTC)), BoundExcluded(datetime(2027, 1, 1, tzinfo=UTC))),
    logo=b"GIF89a",
    tags={"monthly", "promo"},
).save()

basic = await Subscription.objects().get("basic")
for name in Subscription.model_fields:
    value = getattr(basic, name)
    shown = sorted(value) if isinstance(value, set) else value  # a set prints in no fixed order
    print(f"{name:<13} {type_name(value):<27} {shown!s}")

id            builtins.str                basic
started_at    datetime.datetime           2026-01-02 09:30:00+00:00
renews_on     datetime.date               2026-02-02
trial         datetime.timedelta          7 days, 0:00:00
grace         surreal_orm_lite.Duration   1h30m1ns
price         decimal.Decimal             9.99
token         uuid.UUID                   0190c3d1-5c3b-7cc6-8f8a-2d2b1f0e9a11
active_hours  surreal_orm_lite.Range      9..17
period        surreal_orm_lite.Range      d'2026-01-01T00:00:00Z'..d'2027-01-01T00:00:00Z'
logo          builtins.bytes              b'GIF89a'
tags          builtins.set                ['monthly', 'promo']


The raw row, read with the SDK's `client.query()`, holds the SDK's types: its `Duration` and
`Range` classes, a `datetime` for both dates, a `list` for the set. The ORM converts them to the
annotation of each field when it builds the model.

In [4]:
from surrealdb import RecordID

row = await client.query("SELECT * FROM ONLY $rid;", {"rid": RecordID("nb_subscription", "basic")})
for column, value in sorted(row.items()):
    print(f"{column:<13} {type_name(value)}")

active_hours  surrealdb.Range
grace         surrealdb.Duration
id            surrealdb.RecordID
logo          builtins.bytes
period        surrealdb.Range
price         decimal.Decimal
renews_on     datetime.datetime
started_at    datetime.datetime
tags          builtins.list
token         uuid.UUID
trial         surrealdb.Duration


## 4. Precision

SurrealDB durations and datetimes have **nanoseconds**; Python's `timedelta` and `datetime` stop at
microseconds. `Duration` keeps the nanoseconds, and `to_timedelta()` floors them.

In [5]:
grace = Duration.parse("1h30m1ns")
print("Duration:      ", grace, "| elapsed =", grace.elapsed, "ns")
print("to_timedelta():", repr(grace.to_timedelta()), "(floored to the microsecond)")
print("to_compact():  ", grace.to_compact(), "| [seconds, nanoseconds]")

Duration:       1h30m1ns | elapsed = 5400000000001 ns
to_timedelta(): datetime.timedelta(seconds=5400) (floored to the microsecond)
to_compact():   [5400, 1] | [seconds, nanoseconds]


A record written by the server with nanoseconds, here a `d'…05.123456789Z'` literal (deterministic,
unlike `time::now()`): the `Duration` field keeps the nanosecond, the `timedelta` and `datetime`
fields read back truncated to the microsecond.

`new_subscription()` builds the other subscriptions of this notebook: only what each example needs
changes.

In [6]:
def new_subscription(
    sub_id: str,
    trial: timedelta = timedelta(days=1),
    price: Decimal = Decimal("4.99"),
    tags: set[str] | None = None,
) -> Subscription:
    """A subscription with `trial`, `price` and `tags`, the other fields fixed."""
    return Subscription(
        id=sub_id,
        started_at=datetime(2026, 3, 1, tzinfo=UTC),
        renews_on=date(2026, 2, 2),
        trial=trial,
        price=price,
        token=UUID("0190c3d1-5c3b-7cc6-8f8a-2d2b1f0e9a12"),
        active_hours=Range(BoundIncluded(0), BoundExcluded(24)),
        tags=tags or set(),
    )


PRECISE = RecordID("nb_subscription", "precise")
await new_subscription("precise", price=Decimal("1E-7")).save()
await client.query(
    "UPDATE $rid SET started_at = d'2026-01-02T03:04:05.123456789Z', trial = 1h30m1ns, grace = 1h30m1ns;",
    {"rid": PRECISE},
)

precise = await Subscription.objects().get("precise")
print("started_at:", precise.started_at.isoformat(), "(server: …05.123456789)")
print("trial:     ", repr(precise.trial), "(timedelta: the nanosecond is lost)")
print("grace:     ", repr(precise.grace), "(Duration: the nanosecond is kept)")

started_at: 2026-01-02T03:04:05.123456+00:00 (server: …05.123456789)
trial:      datetime.timedelta(seconds=5400) (timedelta: the nanosecond is lost)
grace:      Duration(elapsed=5400000000001) (Duration: the nanosecond is kept)


The truncated datetime is not the stored one, so an `exact` filter with it misses the record.
Filter on a window instead: `gte` the value read, `lt` one microsecond later.

In [7]:
read = precise.started_at
exact = await Subscription.objects().filter(started_at=read).exec()
window = await Subscription.objects().filter(started_at__gte=read, started_at__lt=read + timedelta(microseconds=1)).exec()
print("filter(started_at=read):             ", [s.id for s in exact])
print("filter(started_at__gte=read, __lt=…):", [s.id for s in window])

filter(started_at=read):              []
filter(started_at__gte=read, __lt=…): ['precise']


`precise` was saved with `price=Decimal("1E-7")`. SurrealDB 2.x refuses a decimal written with an
exponent, so the ORM sends it in fixed-point notation (`0.0000001`) on both lines, a filter value
too: the server stores the same value, which reads back equal and matches the filter.

A `Decimal` with trailing zeros reads back as written on SurrealDB 2.x (`12.50`) and normalised on
3.x (`12.5`): the values are equal. The output below comes from SurrealDB 3.x.

In [8]:
print("read back:", repr(precise.price), "| equal to Decimal('1E-7'):", precise.price == Decimal("1E-7"))
matches = await Subscription.objects().filter(price=Decimal("1E-7")).exec()  # sent in fixed point too
print("filter(price=Decimal('1E-7')):", [s.id for s in matches])

await precise.merge(price=Decimal("12.50"))
price = (await Subscription.objects().get("precise")).price
print("12.50:    ", repr(price), "| equal to Decimal('12.50'):", price == Decimal("12.50"))

read back: Decimal('1E-7') | equal to Decimal('1E-7'): True
filter(price=Decimal('1E-7')): ['precise']
12.50:     Decimal('12.5') | equal to Decimal('12.50'): True


## 5. Filters with typed values

A filter value is sent exactly like a field value, so a filter on a rich field compares with the
server's own type: a `timedelta` is a duration, a `date` the datetime at midnight UTC that `save()`
stored. A third subscription gives the filters something to tell apart.

> **Filter values must be typed.** Strings are not converted: `filter(trial__gte="7d")` compares
> the stored duration with the string `"7d"`, which does not give the result you want. Pass
> `timedelta(days=7)`, `date(…)`, `Decimal(…)`, `UUID(…)`.

In [9]:
await Subscription(
    id="pro",
    started_at=datetime(2025, 12, 15, tzinfo=UTC),
    renews_on=date(2026, 12, 15),
    trial=timedelta(days=14),
    price=Decimal("29.99"),
    token=UUID("0190c3d1-5c3b-7cc6-8f8a-2d2b1f0e9a13"),
    active_hours=Range(BoundIncluded(18), BoundIncluded(23)),  # 18..=23
    period=Range(BoundIncluded(datetime(2025, 12, 15, tzinfo=UTC)), None),  # open end
    tags={"yearly"},
).save()


async def ids(**lookups: object) -> list[str]:
    """The ids of the subscriptions matching `lookups`, sorted."""
    return sorted(s.id for s in await Subscription.objects().filter(**lookups).exec())


for label, lookups in [
    ("trial__gte=timedelta(days=7)", {"trial__gte": timedelta(days=7)}),
    ("renews_on=date(2026, 2, 2)", {"renews_on": date(2026, 2, 2)}),
    ("price__lte=Decimal('9.99')", {"price__lte": Decimal("9.99")}),
    ("token=BASIC_TOKEN", {"token": BASIC_TOKEN}),
    ("active_hours__contains=12", {"active_hours__contains": 12}),
]:
    print(f"{label:<29}", await ids(**lookups))

trial__gte=timedelta(days=7)  ['basic', 'pro']
renews_on=date(2026, 2, 2)    ['basic', 'precise']
price__lte=Decimal('9.99')    ['basic']
token=BASIC_TOKEN             ['basic']
active_hours__contains=12     ['basic', 'precise']


`__in` and `__not_in` with a `Range` test membership: the range is bound as one value
(`hour IN $_f0`), never expanded, and an open side works too. A builtin `range` keeps its meaning,
the list of its integers. On the record-id column `id` (and an edge's `in`/`out`) a `Range` raises
`ValueError`: a record id never falls inside a value range.

In [10]:
for hour in (8, 9, 12, 16, 17, 20):
    await Slot(id=f"h{hour}", hour=hour).save()


async def hours(**lookups: object) -> list[int]:
    """The hours of the slots matching `lookups`, in order."""
    return [s.hour for s in await Slot.objects().filter(**lookups).order_by("hour").exec()]


office = Range(BoundIncluded(9), BoundExcluded(17))  # 9..17
evening = Range(begin=BoundIncluded(17))  # 17.., open end
print(f"{'hour__in=' + str(office):<22}", await hours(hour__in=office))
print(f"{'hour__not_in=' + str(evening):<22}", await hours(hour__not_in=evening))
print(f"{'hour__in=range(9, 13)':<22}", await hours(hour__in=range(9, 13)))

hour__in=9..17         [9, 12, 16]
hour__not_in=17..      [8, 9, 12, 16]
hour__in=range(9, 13)  [9, 12]


## 6. Live query

A live query cannot carry bound parameters on SurrealDB 2.x, so the ORM writes each filter value
into the statement as a SurrealQL literal: `trial >= 1w`, not a string. One subscription outside the
filter is saved, then one inside: only the second is reported, as a typed `Subscription`.
`asyncio.timeout` keeps the notebook from hanging on a quiet stream.

In [11]:
import asyncio

async with asyncio.timeout(15):
    async with Subscription.objects().filter(trial__gte=timedelta(days=7)).live() as stream:
        await new_subscription("short", trial=timedelta(days=1)).save()  # outside the filter
        await new_subscription("long", trial=timedelta(days=30)).save()
        event = await anext(stream)

print(event.action, event.record_id)
print("instance: ", type(event.instance).__name__)
print("trial:    ", repr(event.instance.trial))
print("renews_on:", repr(event.instance.renews_on))

CREATE nb_subscription:long
instance:  Subscription
trial:     datetime.timedelta(days=30)
renews_on: datetime.date(2026, 2, 2)


## 7. JSON

`model_dump_json()` writes a `Duration` and a `Range` as their SurrealQL text (`"1h30m1ns"`,
`"18..=23"`), and `model_validate_json()` reads that text back. The other types use Pydantic's
usual JSON forms (a `timedelta` is ISO 8601, a `set` a list).

In [12]:
import json

sample = await Subscription.objects().get("pro")
sample.grace = Duration.parse("1h30m1ns")
dumped = sample.model_dump_json()
print(json.dumps(json.loads(dumped), indent=1))

again = Subscription.model_validate_json(dumped)
print("round trip equal:", again == sample)

{
 "id": "pro",
 "started_at": "2025-12-15T00:00:00Z",
 "renews_on": "2026-12-15",
 "trial": "P14D",
 "grace": "1h30m1ns",
 "price": "29.99",
 "token": "0190c3d1-5c3b-7cc6-8f8a-2d2b1f0e9a13",
 "active_hours": "18..=23",
 "period": "d'2025-12-15T00:00:00Z'..",
 "logo": null,
 "tags": [
  "yearly"
 ]
}
round trip equal: True


## 8. Guards

A value SurrealDB cannot hold is refused before anything is sent, with a plain `ValueError` that
says why; nothing reaches the server, and the same client carries on:

* a **zero duration**, while the installed surrealdb SDK cannot read one back (SDK 2.0.0): store
  `None` instead (declare the field optional) and filter "longer than zero" with
  `__gte=Duration(1)`. The guard lifts by itself with an SDK that reads zero durations, hence the
  two branches below;
* an **integer beyond 64 bits**, which SurrealDB would wrap or reject: use `Decimal`;
* a **decimal beyond the server's range**, an integer part above `2**96 - 1`.

`atomic_increment()` adds numbers only: SurrealDB 2.x ignores `+=` with a duration, so a duration
amount raises `TypeError` on both lines. Set the new value with `merge()` instead.

In [13]:
before = await Subscription.objects().count()
try:
    await new_subscription("zero", trial=timedelta(0)).save()
except ValueError as exc:
    print("zero timedelta:", exc)
else:
    print("zero timedelta: stored (the installed surrealdb SDK reads zero durations back)")

try:
    await Slot(id="huge", hour=2**70).save()
except ValueError as exc:
    print("2**70:", exc)

try:
    await new_subscription("rich", price=Decimal("79228162514264337593543950336")).save()
except ValueError as exc:
    print("Decimal:", exc)

try:
    await basic.atomic_increment("trial", timedelta(days=1))
except TypeError as exc:
    print("atomic_increment:", exc)

await basic.merge(trial=basic.trial + timedelta(days=1))
print("merge():", repr((await Subscription.objects().get("basic")).trial))
print("subscriptions before:", before, "| after:", await Subscription.objects().count())
print("the same client still answers:", await client.query("RETURN 1;", {}))

zero timedelta: Cannot send a zero duration (datetime.timedelta(0)): the installed surrealdb SDK cannot read a zero duration back. Store None instead, or filter with __gte=Duration(1).
2**70: Cannot send 1180591620717411303424: SurrealDB integers are signed 64-bit (-2**63 .. 2**63 - 1); use Decimal for larger numbers.
Decimal: Cannot send Decimal('79228162514264337593543950336'): its integer part, rounded, exceeds 2**96 - 1, the largest SurrealDB decimal.
atomic_increment: atomic_increment() adds a number, not a duration (datetime.timedelta(days=1)): SurrealDB 2.x ignores += with a duration, so the result would differ between server lines. Set the new value with merge(trial=…) instead.


merge(): datetime.timedelta(days=8)
subscriptions before: 5 | after: 5
the same client still answers: 1


## 9. Sets on SurrealDB 3.x

A `set[X]` field reads back as a `set` from a schemaless column (as above) or an `array<…>`
column. A column declared `TYPE set<…>` behaves differently by line: SurrealDB 2.x stores the
array, 3.1 refuses it, and 3.3 stores a native set that the surrealdb SDK 2.0.0 cannot read (every
read of the column fails with an SDK decoding error). So in a schema declare such columns
`array<…>`, with `VALUE $value.distinct()` to keep the items unique on the server too. The cell
below gives the same result on both lines.

In [14]:
await client.query("DEFINE FIELD tags ON nb_subscription TYPE array<string> VALUE $value.distinct();", {})

await new_subscription("team", tags={"ops", "dev", "billing"}).save()
team = await Subscription.objects().get("team")
stored = await client.query("SELECT VALUE tags FROM ONLY $rid;", {"rid": RecordID("nb_subscription", "team")})
print("stored:   ", type_name(stored), stored)
print("read back:", type_name(team.tags), sorted(team.tags))

stored:    builtins.list ['billing', 'dev', 'ops']
read back: builtins.set ['billing', 'dev', 'ops']


## 10. Cleanup

Drop the demo tables and close the connection.

In [15]:
await reset()
await SurrealDBConnectionManager.close_connection()
print("Done")

Done
